# AquaTrack: fine-tune YOLO11n-pose on SwimXYZ (Colab)

Run the cells top to bottom (Shift+Enter).
Before you start: **Runtime > Change runtime type > T4 GPU**.

Everything you want to keep (dataset zip, training runs, `best.pt`) is saved to Google Drive in `MyDrive/aquatrack_swim/`,
because Colab deletes its own disk when the session ends.

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Connect Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DRIVE = Path('/content/drive/MyDrive/aquatrack_swim')
DRIVE.mkdir(parents=True, exist_ok=True)
print('Saving to', DRIVE)

## 3. Install Ultralytics

In [ ]:
!pip install -q ultralytics==8.4.173 requests

## 4. Dataset settings

Same recipe as the Mac version (`pose-practice/swimxyz/build_dataset.py`), with more videos per swimmer.
Change these numbers to make the dataset bigger. If you change them, delete `MyDrive/aquatrack_swim/dataset.zip` so it gets rebuilt.

In [ ]:
VIEWS = ["Side_underwater", "Side_water_level"]   # also available: "Side_above_water", "Front", "Aerial"
VAL_SWIMMER = "Swimmer_Skin_0,75_Muscle_8"         # this swimmer is only used for validation
VIDEOS_PER_SWIMMER = 6    # per view, per swimmer (Mac version used 3)
FRAMES_PER_VIDEO = 20
MIN_VISIBLE = 8           # skip frames with fewer joints inside the image
ZOOM_CROPS = True         # also add zoomed-in crops of the swimmer

## 5. Helper: download only the needed files from the big SwimXYZ zips

In [ ]:
%%writefile /content/remote_zip.py
"""Read single files out of a big zip on a web server with HTTP Range requests."""
import io
import requests


class HttpFile(io.RawIOBase):
    def __init__(self, url):
        self.session = requests.Session()
        r = self.session.head(url, allow_redirects=True, timeout=60)
        r.raise_for_status()
        self.url = r.url
        self.size = int(r.headers["Content-Length"])
        self.pos = 0
        self.downloaded = 0

    def seekable(self):
        return True

    def readable(self):
        return True

    def tell(self):
        return self.pos

    def seek(self, offset, whence=0):
        self.pos = {0: offset, 1: self.pos + offset, 2: self.size + offset}[whence]
        return self.pos

    def read(self, n=-1):
        if n is None or n < 0:
            n = self.size - self.pos
        if n == 0 or self.pos >= self.size:
            return b""
        end = min(self.pos + n, self.size) - 1
        for attempt in range(4):
            try:
                r = self.session.get(self.url, headers={"Range": f"bytes={self.pos}-{end}"}, timeout=60)
                r.raise_for_status()
                if r.status_code != 206:
                    raise RuntimeError("Server does not support Range requests")
                data = r.content
                break
            except Exception:
                if attempt == 3:
                    raise
        self.pos += len(data)
        self.downloaded += len(data)
        return data

    def readinto(self, b):
        data = self.read(len(b))
        b[: len(data)] = data
        return len(data)

## 6. Build the dataset (or reuse the one saved in Drive)

The first run downloads videos from Zenodo and converts the labels to YOLO Pose format. That takes a while.
Later sessions just unzip `dataset.zip` from Drive.

In [ ]:
import io, random, shutil, sys, zipfile
import cv2

sys.path.insert(0, "/content")
from remote_zip import HttpFile

VIDEO_ZIP = "https://zenodo.org/api/records/8402031/files/Freestyle_part2.zip/content"
LABEL_ZIP = "https://zenodo.org/api/records/8399376/files/Freestyle_labels.zip/content"
OUT = Path("/content/dataset")      # local disk: much faster for training than Drive
CACHE = Path("/content/cache")
DATASET_ZIP = DRIVE / "dataset.zip"

# SwimXYZ "COCO" labels use OpenPose's 18-joint order; this maps them to COCO-17 (YOLO Pose)
TO_COCO17 = [0, 15, 14, 17, 16, 5, 2, 6, 3, 7, 4, 11, 8, 12, 9, 13, 10]
FLIP_IDX = [0, 2, 1, 4, 3, 6, 5, 8, 7, 10, 9, 12, 11, 14, 13, 16, 15]
Y_SHIFT = 6  # SwimXYZ y is measured from the bottom; found by overlaying labels on video


def parse_labels(text, height):
    frames = []
    for line in text.splitlines()[1:]:
        if not line.strip():
            continue
        v = [float(x.replace(",", ".")) for x in line.strip(";").split(";")]
        pts = [(v[i * 3], height - v[i * 3 + 1] + Y_SHIFT) for i in range(18)]
        frames.append([pts[i] for i in TO_COCO17])
    return frames


def to_yolo_line(pts, w, h):
    inside = [(x, y) for x, y in pts if 0 <= x < w and 0 <= y < h]
    if len(inside) < MIN_VISIBLE:
        return None
    xs, ys = [p[0] for p in inside], [p[1] for p in inside]
    pad = 0.08 * max(max(xs) - min(xs), max(ys) - min(ys))
    x1, x2 = max(min(xs) - pad, 0), min(max(xs) + pad, w - 1)
    y1, y2 = max(min(ys) - pad, 0), min(max(ys) + pad, h - 1)
    parts = [0, (x1 + x2) / 2 / w, (y1 + y2) / 2 / h, (x2 - x1) / w, (y2 - y1) / h]
    for x, y in pts:
        parts += [x / w, y / h, 2] if (0 <= x < w and 0 <= y < h) else [0, 0, 0]
    return " ".join(f"{p:.6f}" if isinstance(p, float) else str(p) for p in parts)


def zoom_crop(frame, pts):
    h, w = frame.shape[:2]
    inside = [(x, y) for x, y in pts if 0 <= x < w and 0 <= y < h]
    xs, ys = [p[0] for p in inside], [p[1] for p in inside]
    body_w = max(max(xs) - min(xs), 1)
    cw = min(int(body_w / random.uniform(0.5, 0.9)), w)
    ch = min(int(cw * 9 / 16), h)
    cx = (min(xs) + max(xs)) / 2 + random.uniform(-0.1, 0.1) * cw
    cy = (min(ys) + max(ys)) / 2 + random.uniform(-0.15, 0.15) * ch
    x0 = int(min(max(cx - cw / 2, 0), w - cw))
    y0 = int(min(max(cy - ch / 2, 0), h - ch))
    return frame[y0:y0 + ch, x0:x0 + cw], [(x - x0, y - y0) for x, y in pts]


def build():
    random.seed(0)
    CACHE.mkdir(exist_ok=True)
    print("Reading zip listings...")
    vfile, lfile = HttpFile(VIDEO_ZIP), HttpFile(LABEL_ZIP)
    vzip = zipfile.ZipFile(io.BufferedReader(vfile, 1 << 22))
    lzip = zipfile.ZipFile(io.BufferedReader(lfile, 1 << 16))
    videos = [n for n in vzip.namelist() if n.endswith(".webm")]

    chosen = []
    for view in VIEWS:
        swimmers = sorted({n.split("/")[1] for n in videos if n.startswith(view + "/")})
        for swimmer in swimmers:
            pool = [n for n in videos if n.startswith(f"{view}/{swimmer}/")]
            chosen += random.sample(pool, min(VIDEOS_PER_SWIMMER, len(pool)))

    counts = {"train": 0, "val": 0}
    for k, name in enumerate(chosen, 1):
        split = "val" if f"/{VAL_SWIMMER}/" in name else "train"
        label_name = "Freestyle/" + name[: -len(".webm")] + "/COCO/2D_cam.txt"
        local = CACHE / f"video_{k:03d}.webm"
        if not local.exists():
            local.write_bytes(vzip.read(name))
        print(f"[{k}/{len(chosen)}] {split:5s} {name[:70]}  ({vfile.downloaded / 1e6:.0f}MB downloaded)")

        cap = cv2.VideoCapture(str(local))
        w, h = int(cap.get(3)), int(cap.get(4))
        labels = parse_labels(lzip.read(label_name).decode(), h)
        n = min(len(labels), int(cap.get(cv2.CAP_PROP_FRAME_COUNT)))
        img_dir, lbl_dir = OUT / "images" / split, OUT / "labels" / split
        img_dir.mkdir(parents=True, exist_ok=True)
        lbl_dir.mkdir(parents=True, exist_ok=True)

        for fi in sorted(random.sample(range(n), min(FRAMES_PER_VIDEO, n))):
            line = to_yolo_line(labels[fi], w, h)
            if line is None:
                continue
            cap.set(cv2.CAP_PROP_POS_FRAMES, fi)
            ok, frame = cap.read()
            if not ok:
                continue
            stem = f"v{k:03d}_f{fi:03d}"
            cv2.imwrite(str(img_dir / f"{stem}.jpg"), cv2.resize(frame, (960, 540)))
            (lbl_dir / f"{stem}.txt").write_text(line + "\n")
            counts[split] += 1
            if ZOOM_CROPS:
                crop, crop_pts = zoom_crop(frame, labels[fi])
                crop_line = to_yolo_line(crop_pts, crop.shape[1], crop.shape[0])
                if crop_line is not None:
                    cv2.imwrite(str(img_dir / f"{stem}_zoom.jpg"), crop)
                    (lbl_dir / f"{stem}_zoom.txt").write_text(crop_line + "\n")
                    counts[split] += 1
    print(f"Done: {counts['train']} train, {counts['val']} val images, {vfile.downloaded / 1e6:.0f}MB downloaded")


if DATASET_ZIP.exists():
    print("Found dataset.zip in Drive, unzipping...")
    shutil.unpack_archive(DATASET_ZIP, OUT)
else:
    build()
    print("Saving dataset.zip to Drive...")
    shutil.make_archive(str(DATASET_ZIP.with_suffix("")), "zip", OUT)

# data.yaml is always rewritten with this machine's path, so it works in any session
(OUT / "data.yaml").write_text(
    f"path: {OUT}\n"
    "train: images/train\n"
    "val: images/val\n"
    "kpt_shape: [17, 3]\n"
    f"flip_idx: {FLIP_IDX}\n"
    "names:\n  0: person\n"
)
for split in ("train", "val"):
    print(split, len(list((OUT / "images" / split).glob("*.jpg"))), "images")

## 7. Look at a few labels before training

If the dots don't sit on the swimmer's joints, stop here and fix the dataset.

In [ ]:
import matplotlib.pyplot as plt

samples = sorted((OUT / "images" / "train").glob("*.jpg"))[:4]
fig, axes = plt.subplots(1, len(samples), figsize=(5 * len(samples), 4))
for ax, img_path in zip(axes, samples):
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    vals = [float(v) for v in (OUT / "labels" / "train" / (img_path.stem + ".txt")).read_text().split()]
    kpts = vals[5:]
    for i in range(17):
        x, y, v = kpts[i * 3: i * 3 + 3]
        if v > 0:
            ax.plot(x * w, y * h, "o", color="lime", markersize=4)
    ax.imshow(img)
    ax.set_title(img_path.name)
    ax.axis("off")
plt.show()

## 8. Train

Runs are saved straight to Drive (`MyDrive/aquatrack_swim/runs/`), so nothing is lost if Colab disconnects.
`save_period=5` also keeps a checkpoint every 5 epochs.

In [ ]:
from ultralytics import YOLO

RUN_NAME = "swim_colab"

model = YOLO("yolo11n-pose.pt")   # pretrained COCO pose model, downloaded automatically
model.train(
    data=str(OUT / "data.yaml"),
    epochs=50,
    imgsz=640,
    batch=32,
    device=0,          # the Colab GPU
    project=str(DRIVE / "runs"),
    name=RUN_NAME,
    freeze=10,         # keep the first 10 layers fixed so it forgets less
    patience=15,       # stop early if validation stops improving
    save_period=5,
    exist_ok=True,
)

## 8b. Only if Colab disconnected during training

Run cells 1 to 3 and 6 again, then this cell. It continues from the last saved epoch.

In [ ]:
from ultralytics import YOLO

last = DRIVE / "runs" / "swim_colab" / "weights" / "last.pt"
YOLO(str(last)).train(resume=True)

## 9. Results

In [ ]:
import pandas as pd

run = DRIVE / "runs" / "swim_colab"
df = pd.read_csv(run / "results.csv")
df.columns = df.columns.str.strip()
print(df[["epoch", "metrics/mAP50(B)", "metrics/mAP50(P)", "metrics/mAP50-95(P)"]].tail(5).to_string(index=False))
print("\nMac baseline (swim_zoom): pose mAP50 0.786, pose mAP50-95 0.227")
print("Best model:", run / "weights" / "best.pt")

## 10. Try it on your own photo or video

Click the folder icon on the left, upload a file, and put its name below.

In [ ]:
best = YOLO(str(DRIVE / "runs" / "swim_colab" / "weights" / "best.pt"))
results = best.predict("your_photo.jpg", conf=0.25)   # change the file name
plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(results[0].plot(), cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

## 11. Get `best.pt` onto your Mac

It's already in Google Drive at `MyDrive/aquatrack_swim/runs/swim_colab/weights/best.pt`.
Download it from Drive, then on the Mac:

```bash
python pose_photo.py images/person_swimming2.jpeg --model best.pt
```